In [8]:
import os
from dotenv import load_dotenv
from google.cloud import bigquery

In [9]:
load_dotenv()

project_id = os.getenv("GCP_PROJECT_ID")
dataset_id = os.getenv("BQ_DATASET_ID")
credentials_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

print("Project ID:", project_id)
print("Dataset ID:", dataset_id)
print("Credentials:", credentials_path)

Project ID: tc-sql-507706
Dataset ID: tc_sql_dataset
Credentials: ../../credentials/service-account.json


In [10]:
client = bigquery.Client.from_service_account_json(
    credentials_path,
    project=project_id
)

print("Conexión a BigQuery establecida correctamente.")

Conexión a BigQuery establecida correctamente.


In [11]:
dataset_ref = client.dataset(dataset_id)

dataset = client.get_dataset(dataset_ref)

print("Dataset encontrado correctamente:")
print(dataset.dataset_id)

Dataset encontrado correctamente:
tc_sql_dataset


**Creación de tablas**

In [ ]:
#Customers

customers_schema = [
    bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("first_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("last_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("email", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("phone", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("country", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("city", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("acquisition_channel", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("registration_date", "DATE", mode="REQUIRED"),
]

table_ref = f"{project_id}.{dataset_id}.customers"

table = bigquery.Table(table_ref, schema=customers_schema)

table = client.create_table(table, exists_ok=True)

In [20]:
#Categories

categories_schema = [
    bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
]

table_ref = f"{project_id}.{dataset_id}.categories"

table = bigquery.Table(table_ref, schema=categories_schema)

table = client.create_table(table, exists_ok=True)

In [21]:
#Products

products_schema = [
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("sale_price", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("cost_price", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("stock", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("is_active", "BOOL", mode="REQUIRED"),
]

table_ref = f"{project_id}.{dataset_id}.products"

table = bigquery.Table(table_ref, schema=products_schema)

table = client.create_table(table, exists_ok=True)

In [24]:
#Orders 

orders_schema = [
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_status", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("shipping_address", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("shipping_city", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("shipping_country", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("order_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("shipped_date", "DATE", mode="NULLABLE"),
    bigquery.SchemaField("delivered_date", "DATE", mode="NULLABLE"),
]

table_ref = f"{project_id}.{dataset_id}.orders"

table = bigquery.Table(table_ref, schema=orders_schema)

table = client.create_table(table, exists_ok=True)

In [22]:
#Order Items

order_items_schema = [
    bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("quantity", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("unit_price", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("discount", "NUMERIC", mode="REQUIRED"),
]

table_ref = f"{project_id}.{dataset_id}.order_items"

table = bigquery.Table(table_ref, schema=order_items_schema)

table = client.create_table(table, exists_ok=True)

In [25]:
#Payments

payments_schema = [
    bigquery.SchemaField("payment_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("payment_method", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("payment_status", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("amount", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("payment_date", "DATE", mode="REQUIRED"),
]

table_ref = f"{project_id}.{dataset_id}.payments"

table = bigquery.Table(table_ref, schema=payments_schema)

table = client.create_table(table, exists_ok=True)

In [23]:
#Reviews

reviews_schema = [
    bigquery.SchemaField("review_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("rating", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("comment", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("review_date", "DATE", mode="REQUIRED"),
]

table_ref = f"{project_id}.{dataset_id}.reviews"

table = bigquery.Table(table_ref, schema=reviews_schema)

table = client.create_table(table, exists_ok=True)

**Carga de datos en BigQuery**

In [26]:
import os
from google.cloud import bigquery

# Ruta del CSV
csv_path = os.path.join("..", "data", "categories.csv")

# Tabla destino
table_ref = f"{project_id}.{dataset_id}.categories"

# Configuración de carga
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Cargar el CSV
with open(csv_path, "rb") as source_file:
    load_job = client.load_table_from_file(
        source_file,
        table_ref,
        job_config=job_config
    )

# Esperar a que termine
load_job.result()

# Comprobar número de filas
table = client.get_table(table_ref)

print("Carga completada correctamente.")
print("Filas cargadas:", table.num_rows)

Carga completada correctamente.
Filas cargadas: 6


In [27]:
# Tablas que vamos a cargar
tables = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "reviews"
]

# Cargar cada CSV en su tabla correspondiente
for table_name in tables:

    csv_path = os.path.join("..", "data", f"{table_name}.csv")
    table_ref = f"{project_id}.{dataset_id}.{table_name}"

    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    with open(csv_path, "rb") as source_file:
        load_job = client.load_table_from_file(
            source_file,
            table_ref,
            job_config=job_config
        )

    load_job.result()

    table = client.get_table(table_ref)

    print(f"{table_name}: {table.num_rows} filas cargadas")

customers: 500 filas cargadas


BadRequest: 400 Error while reading data, error message: CSV processing encountered too many errors, giving up. Rows: 56; errors: 28; max bad: 0; error percent: 0; reason: invalid, message: Error while reading data, error message: CSV processing encountered too many errors, giving up. Rows: 56; errors: 28; max bad: 0; error percent: 0; reason: invalid, message: Error while reading data, error message: CSV table references column position 7, but line contains only 4 columns.; line_number: 10 byte_offset_to_start_of_line: 1249 column_index: 7 column_name: "is_active" column_type: BOOL; reason: invalid, message: Error while reading data, error message: CSV table references column position 7, but line contains only 4 columns.; line_number: 13 byte_offset_to_start_of_line: 1562 column_index: 7 column_name: "is_active" column_type: BOOL; reason: invalid, message: Error while reading data, error message: CSV table references column position 7, but line contains only 4 columns.; line_number: 1

In [29]:
# Cargar products.csv

csv_path = os.path.join("..", "data", "products.csv")
table_ref = f"{project_id}.{dataset_id}.products"

products_schema = [
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("sale_price", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("cost_price", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("stock", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("is_active", "BOOL", mode="REQUIRED")
]

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    allow_quoted_newlines=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=products_schema
)

with open(csv_path, "rb") as source_file:
    load_job = client.load_table_from_file(
        source_file,
        table_ref,
        job_config=job_config
    )

load_job.result()

table = client.get_table(table_ref)

print("Carga de products completada correctamente.")
print("Filas cargadas:", table.num_rows)

Carga de products completada correctamente.
Filas cargadas: 70


In [30]:
# Cargar los 4 CSV restantes

tables = [
    "orders",
    "order_items",
    "payments",
    "reviews"
]

for table_name in tables:

    csv_path = os.path.join("..", "data", f"{table_name}.csv")
    table_ref = f"{project_id}.{dataset_id}.{table_name}"

    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        allow_quoted_newlines=True,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    with open(csv_path, "rb") as source_file:
        load_job = client.load_table_from_file(
            source_file,
            table_ref,
            job_config=job_config
        )

    load_job.result()

    table = client.get_table(table_ref)

    print(f"{table_name}: {table.num_rows} filas cargadas")

orders: 2000 filas cargadas
order_items: 4500 filas cargadas
payments: 2514 filas cargadas
reviews: 892 filas cargadas
